In [2]:
import nltk

In [3]:
import numpy as np
import json
import glob
import string

In [4]:
#Gensim
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel

In [5]:
#vis
import pyLDAvis
import pyLDAvis.gensim_models

#spacy
import spacy

In [6]:
file_path1 = 'C:/Users/isabe/Documents/unicamp/IC-2024/pesquisa-dos-artigos/education_search_1.ris'
file_path2 = 'C:/Users/isabe/Documents/unicamp/IC-2024/pesquisa-dos-artigos/education_search_2.ris'

def criaArray(path):
    with open(path, 'r', encoding='utf-8') as file:
        lines = file.readlines()
    
    in_abs = False
    abs = []
    count = 0
    
    for line in lines:
        line = line.strip()
        if line.startswith('AB  - '):
            in_abs = True
            abs.append(line[6:])
    
    return abs

abstracts1 = criaArray(file_path1)
abstracts2 = criaArray(file_path2)

abstracts = []
abstracts.extend(abstracts1)
abstracts.extend(abstracts2)

array_unidimensional = np.array(abstracts)

# Transformando em um array bidimensional onde cada elemento seja um array de uma string
array_bidimensional = [[item] for item in array_unidimensional]

primeiros_abs = array_bidimensional[0:2]
print(primeiros_abs)

[['The study’s main purpose was to discover the important factors that impact university students’ online learning and academic performance during the COVID-19 epidemic, as well as their usage of social media throughout the pandemic. Constructivism theory was used and developed with constructs mostly linked to leveraging social media for collaborative learning and student interaction during the COVID-19 pandemic, given the context-dependent nature of online learning during the epidemic. During the COVID-19 epidemic, additional components such as collaborative learning, student participation, and online learning were implemented. The enlarged model, which assesses students’ happiness and academic performance during the COVID-19 epidemic in connection to social media use, was validated using empirical data collected via an online survey questionnaire from 480 Saudi Arabian higher education students. AMOS-SEM was used to analyze the model’s various assumptions (Analysis of Moment Structur

In [7]:
# aplica simple_preprocess: Tokenização, transformação em minúsculas, remoção de acentos
def gen_words(texts):
    final = []
    for array in texts:
        for text in array:
            # Acessa o texto dentro de cada array e aplica simple_preprocess
            new = simple_preprocess(str(text), deacc=True)  # Convertendo para string aqui
            final.append(new)
    return final

data_words = gen_words(array_bidimensional)
print(data_words)

[['the', 'study', 'main', 'purpose', 'was', 'to', 'discover', 'the', 'important', 'factors', 'that', 'impact', 'university', 'students', 'online', 'learning', 'and', 'academic', 'performance', 'during', 'the', 'covid', 'epidemic', 'as', 'well', 'as', 'their', 'usage', 'of', 'social', 'media', 'throughout', 'the', 'pandemic', 'constructivism', 'theory', 'was', 'used', 'and', 'developed', 'with', 'constructs', 'mostly', 'linked', 'to', 'leveraging', 'social', 'media', 'for', 'collaborative', 'learning', 'and', 'student', 'interaction', 'during', 'the', 'covid', 'pandemic', 'given', 'the', 'context', 'dependent', 'nature', 'of', 'online', 'learning', 'during', 'the', 'epidemic', 'during', 'the', 'covid', 'epidemic', 'additional', 'components', 'such', 'as', 'collaborative', 'learning', 'student', 'participation', 'and', 'online', 'learning', 'were', 'implemented', 'the', 'enlarged', 'model', 'which', 'assesses', 'students', 'happiness', 'and', 'academic', 'performance', 'during', 'the', '

In [7]:
print("quantidades de artigos: ", len(data_words))

quant_artigos = 0
for artigos in data_words:
    quant_artigos = quant_artigos + len(artigos)
print("quantidades de tokens artigos total: ", quant_artigos)

quantidades de artigos:  760
quantidades de tokens artigos total:  149957


In [8]:
from nltk.corpus import stopwords
# nltk.download('stopwords') #download só usa vez
stop_words = stopwords.words("english")

def filter_stopwords(words_list):
    filtered_result = []
    
    for sublist in words_list:
        filtered_sublist = []
        for word in sublist:
            if word.lower() not in stop_words:
                filtered_sublist.append(word)
        filtered_result.append(filtered_sublist)
    
    return filtered_result

data_words = filter_stopwords(data_words)
# print(data_words)

In [9]:
# talvez precise instalar a linha a seguir: python -m spacy download en_core_web_sm
def lemmatization(texts, allowed_postags=["NOUN", "ADJ", "VERB", "ADV"]):
    #spacy é um modelo de processamento de linguagem natural. Aqui será usado para o método ".lemma_"
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    output = []
    
    for document in texts:
        lemma_abs = []  # Redefinir lemma_abs a cada iteração
        for word in document:
            doc = nlp(word)
            for token in doc:
                if token.pos_ in allowed_postags:
                    lemma_abs.append(token.lemma_)
    
        output.append(lemma_abs)
    return (output)

data_words = lemmatization(data_words)
# print(data_words)

In [13]:
# corpora.Dictionary elimina palavras repitidas
dictionary = corpora.Dictionary(data_words)
# doc2bow é um método do Dictionary que converte uma lista em  BoW
corpus = [dictionary.doc2bow(doc) for doc in data_words]

# (ID da palavra no dicionário, frequência da palavra no documento)
print(dictionary)

Dictionary<4971 unique tokens: ['academic', 'additional', 'analysis', 'analyze', 'application']...>


In [11]:
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus,
                                           id2word=dictionary,
                                           num_topics= 5,
                                           random_state=100, #semente
                                           update_every=1, #frequência que o modelo é atualizado ao ver cada documento
                                           chunksize=100, #número de documentos a serem usados em cada iteração
                                           passes=10, #número de vezes que o modelo percorrerá o corpus inteiro durante o treinamento
                                           alpha="auto" #distribuição de tópicos por documento
                                           )

In [12]:
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(lda_model, corpus, dictionary, mds="mmds", sort_topics=False)
vis

C:\Users\isabe\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\manifold\_mds.py:298: FutureWarning: The default value of `normalized_stress` will change to `'auto'` in version 1.4. To suppress this warning, manually set the value of `normalized_stress`.
  warnings.warn(


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0      0.104157 -0.127148       1        1  47.706536
1     -0.041619  0.214939       2        1   9.876528
2     -0.111401 -0.213031       3        1  14.075505
3     -0.208585  0.034402       4        1  17.627044
4      0.257448  0.090839       5        1  10.714388, topic_info=        Term         Freq        Total Category  logprob  loglift
71   teacher  2072.000000  2072.000000  Default  30.0000  30.0000
67   student  2315.000000  2315.000000  Default  29.0000  29.0000
42     learn  1672.000000  1672.000000  Default  28.0000  28.0000
52    online   363.000000   363.000000  Default  27.0000  27.0000
532   course   478.000000   478.000000  Default  26.0000  26.0000
..       ...          ...          ...      ...      ...      ...
775  observe    28.808019    47.277199   Topic5  -5.8191   1.7382
68     study    74.788350  1191.673558   Topic5  -4.8651  -0.5349
216    teach    65.206078   823.408688   Topic5  -5.0022  -0.3023
77       use    51.501466   932.365406   Topic5  -5.2381  -0.6625
67   student    49.005031  2315.248463   Topic5  -5.2878  -1.6218

[351 rows x 6 columns], token_table=      Topic      Freq        Term
term                             
0         1  0.092365    academic
0         2  0.281957    academic
0         3  0.063197    academic
0         4  0.559053    academic
788       4  0.961292  acceptance
...     ...       ...         ...
666       1  0.241084       write
666       2  0.427376       write
666       3  0.109584       write
666       4  0.087667       write
666       5  0.131500       write

[626 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[1, 2, 3, 4, 5])